In [2]:
# !pip install PyTDC

     -------------------------------------- 154.2/154.2 KB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 315.1/315.1 KB 6.5 MB/s eta 0:00:00
     -------------------------------------- 542.1/542.1 KB 6.8 MB/s eta 0:00:00
     ---------------------------------------- 84.1/84.1 KB 2.3 MB/s eta 0:00:00
     -------------------------------------- 566.4/566.4 KB 5.9 MB/s eta 0:00:00
     -------------------------------------- 151.3/151.3 KB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.2/151.2 KB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.2/151.2 KB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing m

You should consider upgrading via the 'C:\Users\Veronika\anaconda3\python.exe -m pip install --upgrade pip' command.



     -------------------------------------- 151.1/151.1 KB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.1/151.1 KB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.2/151.2 KB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     -------------------------------------- 151.2/151.2 KB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 151.3/151.3 KB ? eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 2.7/2.7 MB 6.9 MB/s eta 0:00:00
     -----------

In [1]:
from tdc.utils import retrieve_label_name_list
from tdc.single_pred import Tox
from rdkit import Chem
# rdFingerprintGenerator is not automatically exposed as an attribute of Chem in this RDKit installation. 
# Must be imported directly, or 
from rdkit.Chem import rdFingerprintGenerator

import re

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
import rdkit
print(rdkit.__version__)

2022.09.5


In [3]:
label_list = retrieve_label_name_list('Tox21')
label_list

['NR-AR',
 'NR-AR-LBD',
 'NR-AhR',
 'NR-Aromatase',
 'NR-ER',
 'NR-ER-LBD',
 'NR-PPAR-gamma',
 'SR-ARE',
 'SR-ATAD5',
 'SR-HSE',
 'SR-MMP',
 'SR-p53']

## Data loading
Stress-response p53 was selected, as I have an idea what it is responsible for: activation pf the p53 signaling stress-response pathway (carcinogenic). 

The target value is defined for every compound.

Toxicity was found for 423 compounds out of 6774, which means class imbalance 16 : 1.

In [4]:
data_loader = Tox(name = 'Tox21', label_name = 'SR-p53')
df = data_loader.get_data()
display(df.head(3))
print(df.shape)
display(df.isna().sum())
print(df.Y.sum())

Found local copy...
Loading...
Done!


,Drug_ID,Drug,Y
0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,0.0
1,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O,0.0
2,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,0.0


(6774, 3)


Drug_ID    0
Drug       0
Y          0
dtype: int64

423.0


## Data split
I used scaffold split instead of random split as it's claimed to ensure better training. 
Note: shall I test in the future the comparison of performance of models on random splitted data vs scaffold splitted data?

Description: Scaffold split is based on the scaffold of the molecules so that train/val/test set is more structurally different. It is more challenging than random split.

Note: Scaffold split only applies to single-instance drugs-related tasks (ADME, Tox, HTS). Scaffold split also requires RDKit installed.

Sources:
https://tdcommons.ai/functions/data_split/
https://pmc.ncbi.nlm.nih.gov/articles/PMC8627024/

In [5]:
split = data_loader.get_split(method = 'scaffold')
train = split["train"]
valid = split["valid"]
test = split["test"]

100%|████████████████████████████████████████████████████████████████████████████| 6774/6774 [00:02<00:00, 2862.79it/s]


### Split sizes and class balance
From the target values display I see, in the train set 0 is about 20 times more likely than 1, in the validation set 12 times more likely, and in the test set 8.5 more likely. So the class imbalance must be adressed on later steps.

In [6]:
for name, db in split.items():
    print(name)
    print(db.shape)
    temp = db["Y"].value_counts(dropna=False)
    print(temp)
    print("imbalance (times): ", temp[0] / temp[1])
    print()

train
(4741, 3)
0.0    4512
1.0     229
Name: Y, dtype: int64
imbalance (times):  19.70305676855895

valid
(677, 3)
0.0    625
1.0     52
Name: Y, dtype: int64
imbalance (times):  12.01923076923077

test
(1356, 3)
0.0    1214
1.0     142
Name: Y, dtype: int64
imbalance (times):  8.549295774647888



## Convert SMILES to RDKit fingerprints
Sources: https://www.rdkit.org/docs/source/rdkit.Chem.rdmolfiles.html

Check whether I take isotopes into consideration (default setting) or skip it). After a test on the dataset, only 1 compound has an isotopic label, so it can't have an impact. So this parameter can be dropped.

In [7]:
isotope_pattern = re.compile(r"\[[0-9]+[A-Za-z]")

df["has_isotope_notation"] = df["Drug"].str.contains(isotope_pattern, regex=True)

df["has_isotope_notation"].value_counts()

False    6773
True        1
Name: has_isotope_notation, dtype: int64

Machine learning algorithms can only work with number arrays (categorical values would be one-hot encoded) of a fixed length (same length for every row). I didn't find the direct convertion function of SMILES to a numpy array. A way to get around this limitation is to to convert SMILES --> RDKit Molecule object (a molacular graph with nods and edges) --> Morgan fingerprint as an RDKit bit vector of fixed length --> a Numpy array consisting of 0s and 1s for every bit of the BitVector.

In [8]:
def smiles_to_nparr_old(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = Chem.AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    Chem.AllChem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

RDKit GetFingerprintAsNumPy function is documented as returning a NumPy array containing the fingerprint, so this avoids manually allocating arr and calling ConvertToNumpyArray, makes the function faster.

In [9]:
morgan_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_nparr(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return morgan_generator.GetFingerprintAsNumPy(mol)

In [10]:
# New comumn with fingerprints. Can be rerun more than once, doesn't cause an error
# the entry must be a pd.DataFrame
# the entry must contain a column "Drug"
def add_fp_column(df1):
    df1["fingerprint"] = df1["Drug"].apply(smiles_to_nparr)

In [11]:
# the entry must be a pd.DataFrame
# the entry must contain columns "fingerprint" and "Y"
def get_feat_from_nparr(df):
    # 2D NumPy array, every sub-array is a raw of the DataFrame shape (2048, )
    return np.stack(df["fingerprint"].to_numpy())

In [12]:
add_fp_column(train)
add_fp_column(valid)
add_fp_column(test)

In [13]:
# isna() or notna() will also consider None a missing value. - Pandas documentation
print(train.fingerprint.isna().sum())
print(valid.fingerprint.isna().sum())
print(test.fingerprint.isna().sum())

0
0
0


In [14]:
X_train = get_feat_from_nparr(train)
print (X_train.shape, train.Y.shape)

X_valid = get_feat_from_nparr(valid)


(4741, 2048) (4741,)


## Train baseline models
Logistic Regression, Random Forest XGBoost

Handle missing labels and class imbalance: for the logistic rgression, the class imbalance is handled with the inner parameter of the model "class_weight", that will give more weight to the smaller class.

In [16]:
modelLogReg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=12
)

modelLogReg.fit(X_train, train.Y)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=12)

In [21]:
modelRanFor = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

modelRanFor.fit(X_train, train.Y)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

## Predict and evaluate models

PR-AUC depends strongly on class prevalence.

If one endpoint has 20% positives and another has 2% positives, their PR-AUC values are not directly comparable without considering the baseline.

The random baseline for PR-AUC is approximately the positive class fraction:

positive rate = 5%
random PR-AUC ≈ 0.05

In [18]:
y_valid_proba = modelLogReg.predict_proba(X_valid)[:, 1]

In [20]:
roc_auc = roc_auc_score(valid.Y, y_valid_proba)
pr_auc = average_precision_score(valid.Y, y_valid_proba)

print("ROC-AUC Logistic Regression:", roc_auc)
print("PR-AUC Logistic Regression:", pr_auc)

ROC-AUC: 0.7091999999999999
PR-AUC: 0.2340260151179783


In [23]:
y_valid_proba_rf = modelRanFor.predict_proba(X_valid)[:, 1]

roc_auc_rf = roc_auc_score(valid.Y, y_valid_proba_rf)
pr_auc_rf = average_precision_score(valid.Y, y_valid_proba_rf)

print("Random Forest ROC-AUC:", roc_auc_rf)
print("Random Forest PR-AUC:", pr_auc_rf)

Random Forest ROC-AUC: 0.722323076923077
Random Forest PR-AUC: 0.26217658985511716


What to do next:

Figure out why there is separately validation set and test set? Why not do cross validation? Read the TDC documentation

Tune hyperparameters on validation set (if necessary) or using cross-validation on (train concat valid) set then evaluating on test set.

Check out the article https://pmc.ncbi.nlm.nih.gov/articles/PMC8627024/ . Their model is publicly available, I wanna check whether it gives an advanatge in metrics. If the advantage is considerable, dig and understand their approach to train their model (if the code for the training is also publicly available on https://github.com/chen709847237/SSL-GCN).